# YOLO ile Trafik Levhası Tespiti - Model Eğitimi (v3 - poligon etiketleri bbox'a düzeltildi)

Bu notebook, Roboflow'da hazırlanan etiketli veri setini indirir, Ultralytics YOLO ile eğitir ve eğitilmiş `best.pt` modelini Google Drive'a kaydeder.

Sınıflar: `dur`, `yol_ver`, `hiz_siniri`, `park_yasak`, `yaya_gecidi`

Bu sürüm Roboflow'daki **version 3**'ü kullanıyor (v2'deki 337 görüntü ile aynı set, ancak yanlışlıkla poligon olarak kalmış 15 etiket düz bounding box'a çevrilip yeniden export edildi).

**Önemli:** Üstteki menüden `Ortam Zamanı Türü / Runtime type` -> **GPU** (T4) seçili olduğundan emin ol, aksi halde eğitim çok yavaş olur.

## 1) Gerekli kütüphaneleri kur

In [ ]:
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()

## 2) Google Drive'ı bağla (best.pt kalıcı olarak buraya kaydedilecek)

Colab oturumu kapanınca yerel dosyalar silinir. Modelin kaybolmaması için Drive'a bağlanıyoruz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_KAYIT_KLASORU = "/content/drive/MyDrive/YOLO_Trafik_Levhasi"
os.makedirs(DRIVE_KAYIT_KLASORU, exist_ok=True)
print("Kayit klasoru:", DRIVE_KAYIT_KLASORU)

## 3) Roboflow'dan etiketli veri setini indir (version 3 - poligon düzeltmesi sonrası)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="BURAYA_KENDI_API_ANAHTARINIZI_YAZIN")
project = rf.workspace("beyzanur-basaran").project("trafik_levhasi_tespit")
version = project.version(3)
dataset = version.download("yolov8")

print("Veri seti indirildi:", dataset.location)

## 4) data.yaml içeriğini kontrol et

Burada `nc: 5` ve 5 sınıfın doğru listelendiğini görmelisin (`dur, yol_ver, hiz_siniri, park_yasak, yaya_gecidi`).

In [ ]:
with open(f"{dataset.location}/data.yaml", "r") as f:
    print(f.read())

## 5) Modeli eğit

`yolo11n.pt` (nano) - küçük ve hızlı, öğrenci projesi için yeterli. 50 epoch, 640x640 (Roboflow'da da 640x640 export ettik, tutarlı).

Not: bu çalıştırmayı `run3` olarak adlandırıyoruz ki önceki `run1`/`run2` sonuçlarının üzerine yazmasın, karşılaştırabilesin.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="trafik_levhasi_egitim",
    name="run3"
)

## 6) Eğitim sonuçlarını değerlendir (validation seti üzerinde)

mAP50, precision, recall gibi metrikleri gösterir - proje raporunda kullanabilirsin.

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 7) Eğitim grafiklerini göster (kayıp, mAP eğrileri vb.)

Bu görseli proje raporuna ekleyebilirsin. Yol otomatik bulunuyor, elle yol yazmana gerek yok (Colab bazen bunu `runs/detect/...` altına kaydediyor, bazen `trafik_levhasi_egitim/...` altına - ikisini de arıyoruz).

In [ ]:
from IPython.display import Image, display
import glob

def dosya_bul(dosya_adi):
    adaylar = glob.glob(f"**/trafik_levhasi_egitim/run3/{dosya_adi}", recursive=True)
    return adaylar[0] if adaylar else None

results_png = dosya_bul("results.png")
if results_png:
    print("Bulundu:", results_png)
    display(Image(filename=results_png))
else:
    print("results.png bulunamadi")

confusion_png = dosya_bul("confusion_matrix.png")
if confusion_png:
    print("Bulundu:", confusion_png)
    display(Image(filename=confusion_png))
else:
    print("confusion_matrix.png bulunamadi")

## 8) best.pt modelini Google Drive'a kopyala (kalıcı kayıt)

Yol otomatik olarak (glob ile) bulunuyor, hangi klasöre kaydettiyse onu bulup kopyalar.

In [ ]:
import shutil, glob

adaylar = glob.glob("**/trafik_levhasi_egitim/run3/weights/best.pt", recursive=True)
if not adaylar:
    raise FileNotFoundError("best.pt bulunamadi - 5. adimdaki egitim hucresini calistirdigindan emin ol.")

best_pt_kaynak = adaylar[0]
run_klasoru = best_pt_kaynak.replace("/weights/best.pt", "")
print("best.pt bulundu:", best_pt_kaynak)

best_pt_hedef = f"{DRIVE_KAYIT_KLASORU}/best_v3.pt"
shutil.copy(best_pt_kaynak, best_pt_hedef)
print("best.pt su konuma kaydedildi:", best_pt_hedef)

# Tum sonuc klasorunu de (grafikler, metrikler) yedeklemek istersen:
shutil.copytree(run_klasoru, f"{DRIVE_KAYIT_KLASORU}/run3_sonuclar", dirs_exist_ok=True)
print("Tum sonuclar kaydedildi:", f"{DRIVE_KAYIT_KLASORU}/run3_sonuclar")

## 9) best.pt'yi bilgisayarına indir (Drive'a ek olarak)

Bu dosyayı masaüstü uygulamasında kullanacaksın.

In [ ]:
from google.colab import files
files.download(best_pt_kaynak)

## 10) (Opsiyonel) Hızlı test - tek bir görsel üzerinde tahmin dene

In [ ]:
# Ornek: dataset icindeki bir test goruntusu ile hizli deneme
import glob

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
if test_images:
    ornek = test_images[0]
    sonuc = model.predict(source=ornek, save=True, conf=0.25)
    print("Tahmin tamamlandi, sonuc gorseli runs/detect/predict icinde kaydedildi.")
    for r in sonuc:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            conf = float(box.conf[0])
            print(f"Tespit edilen: {cls_name} (guven: {conf:.2f})")